# 2000 年土地覆蓋比例與空間 GP 建模

本分析使用 ESA CCI Land Cover 2000（300 m），計算每個 TCCIP 0.05 度 GRID 內的都市、森林、農業與水域比例。這是以 2000 年為中心的 reference epoch，不是只由 2000 日曆年的單一年影像產生。2000 年土地覆蓋在此是固定的空間解釋變數，不被解釋為 1980--2024 年間皆保持不變。

所有資料下載、像元面積加權與 GP 函式放在 `src/`；本 notebook 只呈現分析順序與結果。

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from land_cover_predictors import build_land_cover_predictors
from land_cover_gp_analysis import (
    TABLE_DIR,
    evaluate_land_cover_return_levels,
    land_cover_parameter_tests,
    load_land_cover_model_data,
    run_land_cover_buffered_cv,
)

## Block 1：建立四種土地覆蓋比例

每個 GRID 以座標為中心建立 0.05 度方格。像元不是只取 GRID 中心點，而是統計整個方格內所有有效像元，並以緯度修正像元面積。

$$
p_{g,c}=\frac{\sum_{j\in g}A_j I(L_j\in c)}{\sum_{j\in g}A_j}.
$$

其中 $p_{g,c}$ 為 GRID $g$ 中類別 $c$ 的比例，$A_j$ 為像元面積權重。

In [ ]:
land_cover = build_land_cover_predictors(force_download=False)
ratio_columns = [
    "urban_ratio",
    "forest_ratio",
    "agriculture_ratio",
    "water_ratio",
    "other_ratio",
]
display(land_cover[ratio_columns].describe())
print("最大加總誤差：", (land_cover["ratio_sum"] - 1).abs().max())

## Block 2：定義候選 GP mean structure

目前模型保留前一階段的 response-specific 結果：

$$
\mu:\ \text{elevation}+\operatorname{GP}_{\mathrm{RBF}},
$$

$$
\log\sigma:\ \text{elevation}+\operatorname{GP}_{\mathrm{Mat\acute ern}(0.5)},
$$

$$
\xi:\ \text{intercept}+\operatorname{GP}_{\mathrm{RBF}}.
$$

土地覆蓋候選模型是在既有 mean structure 後加入 urban、forest、agriculture 與 water proportions。Kernel、五個 geographic folds、buffer distance 與 training cap 保持不變，使差異能歸因於新增 predictor。

In [ ]:
data = load_land_cover_model_data()
fold_metrics, cv_summary, oof_predictions = run_land_cover_buffered_cv(data)
display(cv_summary)

## Block 3：參數別假設檢定

對每個空間 fold 定義：

$$
D_b=\operatorname{MSE}_{\mathrm{current},b}-\operatorname{MSE}_{\mathrm{landcover},b}.
$$

檢定目標為：

$$
H_0:E(D_b)\leq0,
$$

$$
H_1:E(D_b)>0.
$$

使用五個 geographic folds 的單尾 exact sign-flip test。這裡依研究目的分別解讀三個 GEV 參數，所以表內報告 raw p-value。

In [ ]:
parameter_tests, fold_differences = land_cover_parameter_tests(fold_metrics)
display(parameter_tests)
display(fold_differences)

## Block 4：建立 parameter-specific mixed pipeline

只有在 buffered Spatial CV 的 MSE 較低且單尾 raw p-value 小於 0.05 時，才對該參數採用土地覆蓋模型。其他參數保留現有模型。最後使用相同 OOF GRID 的三個參數重新計算：

$$
RL_T=\mu+\frac{\sigma}{\xi}\left[\{-\log(1-1/T)\}^{-\xi}-1\right].
$$

In [ ]:
selected_model_by_target = {
    row.target: (
        "land_cover_2000"
        if row.raw_p < 0.05 and row.mean_MSE_improvement > 0
        else "current"
    )
    for row in parameter_tests.itertuples(index=False)
}
selected_model_by_target

In [ ]:
rl_metrics, rl_tests, rl_oof_predictions = evaluate_land_cover_return_levels(
    oof_predictions,
    selected_model_by_target=selected_model_by_target,
)
display(rl_metrics)
display(rl_tests)

## 解讀限制

- 2000 年土地覆蓋是 reference-year spatial predictor，不是時間變動的 causal exposure。
- `water_ratio` 包含海域，因此沿海或離島 GRID 可能具有很高的水域比例。
- 四個比例屬於 compositional predictors；完整模型保留 `other_ratio` 作為未放入 mean structure 的參考組，避免五類比例全部放入造成完全共線性。
- 五個 folds 的 exact test 解析度有限；$p=0.0625$ 代表證據接近但尚未達 0.05，不等於完全沒有改善。
- 最終結論仍應使用 repeated spatial partitions 檢查 fold partition uncertainty。